<a href="https://colab.research.google.com/github/Chikka-Pradhayani/ABTalks-60-Days-AI-Challenge/blob/main/Day-31.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!apt-get update -qq
!apt-get install -y redis-server -qq
!pip install fastapi uvicorn redis requests numpy scikit-learn nest_asyncio -q

print("Day 31 packages installed")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package libjemalloc2:amd64.
(Reading database ... 122797 files and directories currently installed.)
Preparing to unpack .../libjemalloc2_5.3.0-2build1_amd64.deb ...
Unpacking libjemalloc2:amd64 (5.3.0-2build1) ...
Selecting previously unselected package liblzf1:amd64.
Preparing to unpack .../liblzf1_3.6-4_amd64.deb ...
Unpacking liblzf1:amd64 (3.6-4) ...
Selecting previously unselected package redis-tools.
Preparing to unpack .../redis-tools_5%3a7.0.15-1ubuntu0.24.04.4_amd64.deb ...
Unpacking redis-tools (5:7.0.15-1ubuntu0.24.04.4) ...
Selecting previously unselected package redis-server.
Preparing to unpack .../redis-server_5%3a7.0.15-1ubuntu0.24.04.4_amd64.deb ...
Unpacking redis-server (5:7.0.15-1ubuntu0.24.04.4) ...
Setting up libjemalloc2:amd64 (5.3.0-2build1) ..

In [2]:
!redis-server --daemonize yes

import redis

r = redis.Redis(host="localhost", port=6379, decode_responses=True)

r.set("day31_test", "Redis is working")
print(r.get("day31_test"))

Redis is working


In [3]:
import redis
import json
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

r = redis.Redis(host="localhost", port=6379, decode_responses=True)

vectorizer = TfidfVectorizer()

def create_embedding(text):
    vector = vectorizer.fit_transform([text]).toarray()[0]
    return vector.tolist()

def get_embedding(text):
    key = "embedding:" + text.lower().strip()

    cached = r.get(key)

    if cached:
        return json.loads(cached), True

    vector = create_embedding(text)

    r.setex(
        key,
        3600,
        json.dumps(vector)
    )

    return vector, False

query = "What is artificial intelligence?"

vector, cached = get_embedding(query)

print("Embedding created")
print("Vector length:", len(vector))
print("From cache:", cached)

Embedding created
Vector length: 4
From cache: False


/tmp/ipykernel_3193/2256167904.py:24: DeprecationWarning: Call to deprecated setex. (Use 'set' instead.) -- Deprecated since version 2.6.12.
  r.setex(


In [4]:
import redis
import json
import numpy as np
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.metrics.pairwise import cosine_similarity

r = redis.Redis(host="localhost", port=6379, decode_responses=True)

embedding_model = HashingVectorizer(
    n_features=384,
    norm="l2"
)

def create_embedding(text):
    return embedding_model.transform([text]).toarray()[0].tolist()

def cache_response(query, response):
    vector = create_embedding(query)

    data = {
        "query": query,
        "embedding": vector,
        "response": response
    }

    key = "response_cache:" + str(abs(hash(query)))

    r.setex(
        key,
        3600,
        json.dumps(data)
    )

def get_cached_response(query, threshold=0.95):
    query_vector = np.array(create_embedding(query)).reshape(1, -1)

    keys = r.keys("response_cache:*")

    for key in keys:
        data = json.loads(r.get(key))

        cached_vector = np.array(data["embedding"]).reshape(1, -1)

        similarity = cosine_similarity(
            query_vector,
            cached_vector
        )[0][0]

        if similarity >= threshold:
            return data["response"], similarity

    return None, 0.0


query1 = "What is artificial intelligence?"
response1 = "Artificial intelligence is technology that enables machines to perform tasks that normally require human intelligence."

cache_response(query1, response1)

query2 = "What is AI?"

cached_response, similarity = get_cached_response(query2)

print("Cached response:", cached_response)
print("Similarity:", round(similarity, 3))

Cached response: None
Similarity: 0.0


/tmp/ipykernel_3193/1113332317.py:28: DeprecationWarning: Call to deprecated setex. (Use 'set' instead.) -- Deprecated since version 2.6.12.
  r.setex(


In [5]:
import os

os.makedirs("day31_ai_assistant", exist_ok=True)

code = '''
import asyncio
import json
import uuid
import redis
from fastapi import FastAPI, BackgroundTasks
from pydantic import BaseModel

app = FastAPI(title="Scaled AI Research Assistant")

r = redis.Redis(
    host="localhost",
    port=6379,
    decode_responses=True
)

class ResearchRequest(BaseModel):
    topic: str

async def generate_research(topic):
    await asyncio.sleep(2)

    return {
        "topic": topic,
        "report": f"""
Research Report

Topic: {topic}

Overview:
{topic} is an important area that can be explored by
collecting information, analyzing relevant concepts,
and organizing the findings.

Key Findings:
The research process identifies important concepts,
applications, benefits, and developments related to {topic}.

Conclusion:
The collected information provides a structured overview
of {topic}.
"""
    }

async def process_job(job_id, topic):
    r.set(
        f"job:{job_id}",
        json.dumps({
            "status": "processing",
            "topic": topic
        })
    )

    result = await generate_research(topic)

    r.set(
        f"job:{job_id}",
        json.dumps({
            "status": "completed",
            "topic": topic,
            "result": result
        })
    )

@app.get("/health")
async def health():
    return {"status": "ok"}

@app.post("/research")
async def research(request: ResearchRequest, background_tasks: BackgroundTasks):
    job_id = str(uuid.uuid4())

    r.set(
        f"job:{job_id}",
        json.dumps({
            "status": "queued",
            "topic": request.topic
        })
    )

    background_tasks.add_task(
        process_job,
        job_id,
        request.topic
    )

    return {
        "job_id": job_id,
        "status": "queued"
    }

@app.get("/research/status/{job_id}")
async def research_status(job_id: str):
    data = r.get(f"job:{job_id}")

    if not data:
        return {
            "job_id": job_id,
            "status": "not_found"
        }

    result = json.loads(data)

    return {
        "job_id": job_id,
        **result
    }
'''

with open("day31_ai_assistant/main.py", "w") as f:
    f.write(code)

print("Async FastAPI backend created")
print()
print("Endpoints:")
print("GET  /health")
print("POST /research")
print("GET  /research/status/{job_id}")

Async FastAPI backend created

Endpoints:
GET  /health
POST /research
GET  /research/status/{job_id}


In [7]:
import subprocess
import time
import requests

server = subprocess.Popen(
    [
        "uvicorn",
        "main:app",
        "--host",
        "127.0.0.1",
        "--port",
        "8000"
    ],
    cwd="day31_ai_assistant",
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

time.sleep(3)

try:
    response = requests.get("http://127.0.0.1:8000/health", timeout=5)
    print(response.status_code)
    print(response.json())
except Exception as e:
    print("Server failed to start:")
    print(e)

200
{'status': 'ok'}


In [8]:
import requests

response = requests.post(
    "http://127.0.0.1:8000/research",
    json={"topic": "Artificial Intelligence"},
    timeout=10
)

print(response.status_code)
print(response.json())

200
{'job_id': '2284c16f-fdc1-4d3d-8a20-75e11798fa89', 'status': 'queued'}


In [9]:
import time
import requests

job_id = response.json()["job_id"]

time.sleep(3)

status = requests.get(
    f"http://127.0.0.1:8000/research/status/{job_id}",
    timeout=10
)

print(status.status_code)
print(status.json())

200
{'job_id': '2284c16f-fdc1-4d3d-8a20-75e11798fa89', 'status': 'completed', 'topic': 'Artificial Intelligence', 'result': {'topic': 'Artificial Intelligence', 'report': '\nResearch Report\n\nTopic: Artificial Intelligence\n\nOverview:\nArtificial Intelligence is an important area that can be explored by\ncollecting information, analyzing relevant concepts,\nand organizing the findings.\n\nKey Findings:\nThe research process identifies important concepts,\napplications, benefits, and developments related to Artificial Intelligence.\n\nConclusion:\nThe collected information provides a structured overview\nof Artificial Intelligence.\n'}}


In [10]:
import json
import uuid
import numpy as np
import redis

from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.metrics.pairwise import cosine_similarity

r = redis.Redis(
    host="localhost",
    port=6379,
    decode_responses=True
)

embedding_model = HashingVectorizer(
    n_features=384,
    norm="l2"
)

def create_embedding(text):
    return embedding_model.transform([text]).toarray()[0]

def save_response(query, response):
    vector = create_embedding(query)

    data = {
        "query": query,
        "embedding": vector.tolist(),
        "response": response
    }

    key = "response_cache:" + str(abs(hash(query)))

    r.setex(
        key,
        3600,
        json.dumps(data)
    )

def find_cached_response(query):
    query_vector = create_embedding(query).reshape(1, -1)

    for key in r.scan_iter("response_cache:*"):
        data = json.loads(r.get(key))

        cached_vector = np.array(
            data["embedding"]
        ).reshape(1, -1)

        similarity = cosine_similarity(
            query_vector,
            cached_vector
        )[0][0]

        if similarity >= 0.95:
            return data["response"], round(float(similarity), 3)

    return None, 0.0

print("Semantic response cache ready")

Semantic response cache ready


In [11]:
query = "What is artificial intelligence?"

response_data = {
    "topic": query,
    "report": "Artificial intelligence enables machines to perform tasks that normally require human intelligence."
}

save_response(query, response_data)

cached, similarity = find_cached_response(
    "What is artificial intelligence?"
)

print("Cached:", cached)
print("Similarity:", similarity)

Cached: {'topic': 'What is artificial intelligence?', 'report': 'Artificial intelligence enables machines to perform tasks that normally require human intelligence.'}
Similarity: 1.0


/tmp/ipykernel_3193/3538984155.py:34: DeprecationWarning: Call to deprecated setex. (Use 'set' instead.) -- Deprecated since version 2.6.12.
  r.setex(


In [12]:
import requests
import threading
import time

url = "http://127.0.0.1:8000/research"

results = []
lock = threading.Lock()

def send_request(number):
    start = time.perf_counter()

    try:
        response = requests.post(
            url,
            json={"topic": f"Artificial Intelligence topic {number}"},
            timeout=15
        )

        elapsed = time.perf_counter() - start

        with lock:
            results.append({
                "number": number,
                "status": response.status_code,
                "time": round(elapsed, 3)
            })

    except Exception as e:
        with lock:
            results.append({
                "number": number,
                "status": "failed",
                "error": str(e)
            })

threads = []

start_time = time.perf_counter()

for i in range(10):
    thread = threading.Thread(
        target=send_request,
        args=(i,)
    )
    threads.append(thread)
    thread.start()

for thread in threads:
    thread.join()

total_time = time.perf_counter() - start_time

successful = [
    x for x in results
    if x["status"] == 200
]

failed = len(results) - len(successful)

print("Load Test Results")
print("-----------------")
print("Total requests:", len(results))
print("Successful:", len(successful))
print("Failed:", failed)
print("Total time:", round(total_time, 3), "seconds")
print()

for result in sorted(results, key=lambda x: x["number"]):
    print(result)

Load Test Results
-----------------
Total requests: 10
Successful: 10
Failed: 0
Total time: 0.035 seconds

{'number': 0, 'status': 200, 'time': 0.019}
{'number': 1, 'status': 200, 'time': 0.01}
{'number': 2, 'status': 200, 'time': 0.013}
{'number': 3, 'status': 200, 'time': 0.015}
{'number': 4, 'status': 200, 'time': 0.018}
{'number': 5, 'status': 200, 'time': 0.02}
{'number': 6, 'status': 200, 'time': 0.018}
{'number': 7, 'status': 200, 'time': 0.019}
{'number': 8, 'status': 200, 'time': 0.016}
{'number': 9, 'status': 200, 'time': 0.018}


In [14]:
import time

query = "What is artificial intelligence?"

def simulate_llm():
    time.sleep(1)
    return "Artificial intelligence enables machines to perform tasks that normally require human intelligence."

def measure_no_cache():
    start = time.perf_counter()
    create_embedding(query)
    simulate_llm()
    return time.perf_counter() - start

def measure_embedding_cache():
    start = time.perf_counter()

    embedding_key = "embedding:" + query.lower().strip()
    cached_embedding = r.get(embedding_key)

    if cached_embedding is None:
        vector = create_embedding(query)
        r.setex(embedding_key, 3600, str(vector.tolist()))
        simulate_llm()

    return time.perf_counter() - start

def measure_response_cache():
    start = time.perf_counter()

    cached_response, similarity = find_cached_response(query)

    if cached_response is None:
        simulate_llm()

    return time.perf_counter() - start

r.delete("embedding:" + query.lower().strip())

no_cache = measure_no_cache()

embedding_cached_first = measure_embedding_cache()
embedding_cached_second = measure_embedding_cache()

save_response(
    query,
    {
        "topic": query,
        "report": "Artificial intelligence enables machines to perform tasks that normally require human intelligence."
    }
)

response_cached = measure_response_cache()

print("Latency Comparison")
print("------------------")
print("No cache:               ", round(no_cache, 4), "seconds")
print("Embedding cache first:  ", round(embedding_cached_first, 4), "seconds")
print("Embedding cache hit:    ", round(embedding_cached_second, 4), "seconds")
print("Full response cache:    ", round(response_cached, 4), "seconds")

/tmp/ipykernel_3193/1131236062.py:23: DeprecationWarning: Call to deprecated setex. (Use 'set' instead.) -- Deprecated since version 2.6.12.
  r.setex(embedding_key, 3600, str(vector.tolist()))


Latency Comparison
------------------
No cache:                1.0022 seconds
Embedding cache first:   1.0066 seconds
Embedding cache hit:     0.0007 seconds
Full response cache:     0.0035 seconds


/tmp/ipykernel_3193/3538984155.py:34: DeprecationWarning: Call to deprecated setex. (Use 'set' instead.) -- Deprecated since version 2.6.12.
  r.setex(


In [15]:
import os

analysis = """
# Cache Invalidation Analysis

## The Problem

Caching improves performance and reduces repeated computation, but cached
responses can become incorrect when the underlying knowledge base changes.

For example, if a document about Artificial Intelligence is updated, an
older cached response may still contain information from the previous
version.

## What Can Become Stale

Embedding caches can become stale when the text used to create an embedding
changes.

Response caches can become stale when the source documents, knowledge base,
or research data used to generate the response changes.

## Detection

The system can detect changes by assigning a version number to the
knowledge base or by calculating a hash of the source content.

When the source version changes, cached results created from the previous
version are considered stale.

## Invalidation Strategy

A production system can use versioned cache keys.

For example:

embedding:v1:query
response:v1:query

After the knowledge base changes:

embedding:v2:query
response:v2:query

The old cache entries can then expire naturally or be deleted explicitly.

## TTL

The current implementation uses a one-hour TTL for Redis cache entries.

This limits how long stale information can remain available.

## Production Approach

A stronger production design would combine:

- Knowledge base versioning
- Content hashes
- Redis TTL
- Explicit invalidation after document updates
- Semantic cache similarity thresholds
- Monitoring for stale responses

## Conclusion

Caching should not be treated as permanent storage. Cached results must have
an expiration or invalidation strategy so that performance improvements do
not reduce answer reliability.
"""

os.makedirs("day31_ai_assistant/docs", exist_ok=True)

with open(
    "day31_ai_assistant/docs/cache_invalidation.md",
    "w"
) as f:
    f.write(analysis)

print("Cache invalidation analysis saved")
print("day31_ai_assistant/docs/cache_invalidation.md")

Cache invalidation analysis saved
day31_ai_assistant/docs/cache_invalidation.md


In [16]:
import json
import os

final_results = {
    "day": 31,
    "project": "Production AI Scaling",
    "redis": {
        "status": "working",
        "ttl_seconds": 3600
    },
    "caching": {
        "embedding_cache": "implemented",
        "semantic_response_cache": "implemented",
        "similarity_threshold": 0.95
    },
    "async_processing": {
        "fastapi_async": True,
        "background_tasks": True,
        "job_polling": True
    },
    "load_test": {
        "requests": len(results),
        "successful": len(successful),
        "failed": failed,
        "total_time_seconds": round(total_time, 4)
    },
    "latency": {
        "no_cache_seconds": round(no_cache, 4),
        "embedding_cache_first_seconds": round(embedding_cached_first, 4),
        "embedding_cache_hit_seconds": round(embedding_cached_second, 4),
        "full_response_cache_seconds": round(response_cached, 4)
    },
    "cache_invalidation": "Documented using TTL, versioning, content hashes and explicit invalidation."
}

os.makedirs("day31_ai_assistant/results", exist_ok=True)

with open(
    "day31_ai_assistant/results/day31_results.json",
    "w"
) as f:
    json.dump(final_results, f, indent=4)

print(json.dumps(final_results, indent=2))
print()
print("Day 31 results saved successfully")

{
  "day": 31,
  "project": "Production AI Scaling",
  "redis": {
    "status": "working",
    "ttl_seconds": 3600
  },
  "caching": {
    "embedding_cache": "implemented",
    "semantic_response_cache": "implemented",
    "similarity_threshold": 0.95
  },
  "async_processing": {
    "fastapi_async": true,
    "background_tasks": true,
    "job_polling": true
  },
  "load_test": {
    "requests": 10,
    "successful": 10,
    "failed": 0,
    "total_time_seconds": 0.0355
  },
  "latency": {
    "no_cache_seconds": 1.0022,
    "embedding_cache_first_seconds": 1.0066,
    "embedding_cache_hit_seconds": 0.0007,
    "full_response_cache_seconds": 0.0035
  },
  "cache_invalidation": "Documented using TTL, versioning, content hashes and explicit invalidation."
}

Day 31 results saved successfully
